# M3 · Three fitted Bayesian MMMs
This notebook uses the **M2B source-derived panel** and saved PyMC posterior artifacts. Run `uv run mmm-fit` from `modeling/` first, or let the fitting cell below do it. The synthetic truth file is never read during fitting. The model equation, priors and limitations are documented in `docs/MODEL_SPECIFICATION.md`.

## Equation and priors
For each channel: `adstock[t] = spend[t]/1000 + alpha × adstock[t−1]`, then `saturated = adstock/(adstock + half_saturation)`. Expected sales in units are `100 × {intercept[segment] × (1 + controls · coefficients) + Σ beta[segment,channel] × saturated}`. The likelihood is Normal with segment-specific noise. Complete pooling shares each channel beta; no pooling gives six independent responses; partial pooling gives log beta a shared channel population and dispersion. Alpha and half-saturation are shared **across segments** and inferred in all three models. Controls are price, distribution, competitor, trend, 52-week sine/cosine seasonality and holiday. The Normal likelihood approximates rounded nonnegative counts.

In [ ]:
import json
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from mmm_lab.model import POOLING, build_model, fit_models, load_model_data
from mmm_lab.paths import ARTIFACTS_DIR

data = load_model_data()
print('Verified M2B input:', data.input_sha256, 'train/holdout weeks:', data.train_weeks, len(data.weeks)-data.train_weeks)
print('Array dimensions:', data.sales.shape, data.spend.shape, data.controls.shape)

In [ ]:
# This is the same reproducible package function used by the mmm-fit CLI.
manifest_path = ARTIFACTS_DIR / 'fit_manifest.json'
if not manifest_path.exists():
    fit_models()  # full MCMC; this can take several minutes
manifest = json.loads(manifest_path.read_text())
assert manifest['model_input_sha256'] == data.input_sha256
display(pd.DataFrame(manifest['files']).T)

## Prior predictive and posterior trace
Prior predictive samples are generated before observing sales. Broad priors can generate unrealistic negative Normal outcomes; inspect that explicitly instead of treating the prior as calibrated. Posterior trace plots are diagnostic views, not evidence of causal identification.

In [ ]:
partial = az.from_netcdf(ARTIFACTS_DIR / manifest['files']['partial']['path'])
prior_sales = partial['prior_predictive']['sales'].values.ravel() * 100
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(prior_sales, bins=40)
ax.set(xlabel='Prior predictive sales units', ylabel='Frequency', title='Partial pooling prior predictive')
plt.show()
print('Negative prior predictive fraction:', np.mean(prior_sales < 0))
display(az.summary(partial, var_names=['adstock_alpha', 'half_saturation', 'beta', 'noise'], kind='stats'))

In [ ]:
az.plot_trace(partial, var_names=['adstock_alpha', 'half_saturation'])
plt.show()
print('Divergent transitions:', int(partial['sample_stats']['diverging'].sum()))

Promotional contributions are **model-based, assumption-dependent estimates**. Correlated schedules, price targeting, and the small segments can leave individual channel effects weakly identified. Evaluate all three specifications and recovery against hidden simulation truth in notebook 03; do not infer experiment-validated incrementality from posterior fit alone.